In [1]:
# ============================================================
# EXPERIMENT 08: SUBJECT-LEVEL AGGREGATION COMPARISON
# ============================================================

print("Experiment 08: Subject-Level Aggregation Comparison")
print()
print("Model: SqueezeNet")
print("XAI Method: Integrated Gradients")
print("Slice Selection: Regional / Diverse Selection")
print("Slices per Subject: 10")
print("Total Selected Slices: 60")
print()
print("Aggregation methods to compare:")
print("1. Mean")
print("2. Median")
print("3. Majority Voting")
print("4. Top-Confidence Mean")
print("5. Trimmed Mean")

Experiment 08: Subject-Level Aggregation Comparison

Model: SqueezeNet
XAI Method: Integrated Gradients
Slice Selection: Regional / Diverse Selection
Slices per Subject: 10
Total Selected Slices: 60

Aggregation methods to compare:
1. Mean
2. Median
3. Majority Voting
4. Top-Confidence Mean
5. Trimmed Mean


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ============================================================
# CELL 2 — LOAD EXPERIMENT 07 SELECTED SLICES
# ============================================================

import pandas as pd
import numpy as np

EXP7_DIR = "/content/drive/MyDrive/ASD_XAI_Project/experiment_07_ig_regional_selection"

SELECTED_PATH = f"{EXP7_DIR}/regional_ig_selected_slices.csv"

selected_df = pd.read_csv(SELECTED_PATH)

In [4]:
print("✅ Experiment 07 selected slices loaded!")
print()
print("Total selected slices:", len(selected_df))
print("Number of subjects:", selected_df["SUB_ID"].nunique())
print()

print("Slices per subject:")
print(selected_df.groupby("SUB_ID").size())

print()
print("Columns:")
print(selected_df.columns.tolist())

✅ Experiment 07 selected slices loaded!

Total selected slices: 60
Number of subjects: 6

Slices per subject:
SUB_ID
51460    10
51463    10
51466    10
51475    10
51476    10
51487    10
dtype: int64

Columns:
['SUB_ID', 'SLICE_PATH', 'LABEL', 'IG_SCORE', 'IG_RANK', 'SLICE_INDEX', 'REGION']


In [5]:
# ============================================================
# CELL 3 — LOAD SQUEEZENET MODEL
# ============================================================

import tensorflow as tf

MODEL_PATH = "/content/drive/MyDrive/ASD_XAI_Project/models/squeezenet_baseline.keras"

squeezenet_model = tf.keras.models.load_model(MODEL_PATH)

IMG_SIZE = 224

def load_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32)
    return image

print("✅ SqueezeNet model loaded!")
print("Input shape:", squeezenet_model.input_shape)
print("Output shape:", squeezenet_model.output_shape)

✅ SqueezeNet model loaded!
Input shape: (None, 224, 224, 3)
Output shape: (None, 1)


In [6]:
# ============================================================
# CELL 4 — GENERATE SLICE-LEVEL PREDICTIONS
# ============================================================

print("Generating predictions for 60 selected slices...")
print("⏳ This may take a little while...")

# Load all 60 images using the exact Experiment 07 preprocessing
images = np.stack([
    load_image(path).numpy()
    for path in selected_df["SLICE_PATH"]
])

Generating predictions for 60 selected slices...
⏳ This may take a little while...


In [7]:
# Predict
slice_predictions = squeezenet_model.predict(
    images,
    batch_size=16,
    verbose=1
).reshape(-1)

# Add predictions to the dataframe
predictions_df = selected_df.copy()
predictions_df["SLICE_PROBABILITY"] = slice_predictions

# Save immediately
PREDICTIONS_PATH = f"{EXP7_DIR}/experiment_08_slice_predictions.csv"

predictions_df.to_csv(
    PREDICTIONS_PATH,
    index=False
)

print()
print("✅ Predictions generated and saved!")
print("Total predictions:", len(predictions_df))
print("Saved to:", PREDICTIONS_PATH)

print()
print("First 10 predictions:")
display(
    predictions_df[
        ["SUB_ID", "SLICE_INDEX", "REGION", "LABEL", "SLICE_PROBABILITY"]
    ].head(10)
)

4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 653ms/step

✅ Predictions generated and saved!
Total predictions: 60
Saved to: /content/drive/MyDrive/ASD_XAI_Project/experiment_07_ig_regional_selection/experiment_08_slice_predictions.csv

First 10 predictions:


,SUB_ID,SLICE_INDEX,REGION,LABEL,SLICE_PROBABILITY
0,51460,16,1,ASD,0.508340
1,51460,26,2,ASD,0.462786
2,51460,35,3,ASD,0.468560
3,51460,52,4,ASD,0.484365
4,51460,70,5,ASD,0.504124
5,51460,91,6,ASD,0.507811
6,51460,120,7,ASD,0.493372
7,51460,139,8,ASD,0.451121
8,51460,148,9,ASD,0.449729
9,51460,158,10,ASD,0.514695


In [8]:
# ============================================================
# CELL 5 — LOAD SAVED SLICE PREDICTIONS
# ============================================================

PREDICTIONS_PATH = f"{EXP7_DIR}/experiment_08_slice_predictions.csv"

predictions_df = pd.read_csv(PREDICTIONS_PATH)

print("✅ Saved slice predictions loaded!")
print()
print("Total predictions:", len(predictions_df))
print("Subjects:", predictions_df["SUB_ID"].nunique())
print()

print("Predictions per subject:")
print(predictions_df.groupby("SUB_ID").size())

print()
print("Probability range:")
print(
    "Minimum:", predictions_df["SLICE_PROBABILITY"].min()
)
print(
    "Maximum:", predictions_df["SLICE_PROBABILITY"].max()
)

print()
display(
    predictions_df[
        ["SUB_ID", "SLICE_INDEX", "REGION", "LABEL", "SLICE_PROBABILITY"]
    ].head(10)
)

✅ Saved slice predictions loaded!

Total predictions: 60
Subjects: 6

Predictions per subject:
SUB_ID
51460    10
51463    10
51466    10
51475    10
51476    10
51487    10
dtype: int64

Probability range:
Minimum: 0.43595648
Maximum: 0.52310133



,SUB_ID,SLICE_INDEX,REGION,LABEL,SLICE_PROBABILITY
0,51460,16,1,ASD,0.508340
1,51460,26,2,ASD,0.462786
2,51460,35,3,ASD,0.468560
3,51460,52,4,ASD,0.484365
4,51460,70,5,ASD,0.504124
5,51460,91,6,ASD,0.507811
6,51460,120,7,ASD,0.493372
7,51460,139,8,ASD,0.451121
8,51460,148,9,ASD,0.449729
9,51460,158,10,ASD,0.514695


In [9]:
# ============================================================
# CELL 6 — MEAN VS MEDIAN AGGREGATION
# ============================================================

results_mean_median = []

for sub_id, group in predictions_df.groupby("SUB_ID"):

    probabilities = group["SLICE_PROBABILITY"].values
    true_label = group["LABEL"].iloc[0]

    # Mean aggregation
    mean_probability = np.mean(probabilities)
    mean_prediction = "ASD" if mean_probability >= 0.5 else "Control"

    # Median aggregation
    median_probability = np.median(probabilities)
    median_prediction = "ASD" if median_probability >= 0.5 else "Control"

    results_mean_median.append({
        "SUB_ID": sub_id,
        "TRUE_LABEL": true_label,
        "MEAN_PROBABILITY": mean_probability,
        "MEAN_PREDICTION": mean_prediction,
        "MEDIAN_PROBABILITY": median_probability,
        "MEDIAN_PREDICTION": median_prediction
    })

mean_median_df = pd.DataFrame(results_mean_median)

print("✅ Mean vs Median results:")
display(mean_median_df)

✅ Mean vs Median results:


,SUB_ID,TRUE_LABEL,MEAN_PROBABILITY,MEAN_PREDICTION,MEDIAN_PROBABILITY,MEDIAN_PREDICTION
0,51460,ASD,0.484490,Control,0.488868,Control
1,51463,ASD,0.501792,ASD,0.509452,ASD
2,51466,ASD,0.496756,Control,0.494682,Control
3,51475,Control,0.479290,Control,0.480482,Control
4,51476,Control,0.476613,Control,0.485496,Control
5,51487,Control,0.491508,Control,0.492974,Control


In [10]:
# ============================================================
# CELL 7 — MAJORITY VOTING
# ============================================================

majority_results = []

for sub_id, group in predictions_df.groupby("SUB_ID"):

    probabilities = group["SLICE_PROBABILITY"].values
    true_label = group["LABEL"].iloc[0]

    # Convert each slice probability into a slice-level prediction
    slice_predictions = np.where(
        probabilities >= 0.5,
        "ASD",
        "Control"
    )

    # Count votes
    asd_votes = np.sum(slice_predictions == "ASD")
    control_votes = np.sum(slice_predictions == "Control")

    # Majority decision
    if asd_votes > control_votes:
        majority_prediction = "ASD"
    else:
        majority_prediction = "Control"

    majority_results.append({
        "SUB_ID": sub_id,
        "TRUE_LABEL": true_label,
        "ASD_VOTES": asd_votes,
        "CONTROL_VOTES": control_votes,
        "MAJORITY_PREDICTION": majority_prediction
    })

majority_df = pd.DataFrame(majority_results)

print("✅ Majority voting results:")
display(majority_df)

✅ Majority voting results:


,SUB_ID,TRUE_LABEL,ASD_VOTES,CONTROL_VOTES,MAJORITY_PREDICTION
0,51460,ASD,4,6,Control
1,51463,ASD,6,4,ASD
2,51466,ASD,4,6,Control
3,51475,Control,1,9,Control
4,51476,Control,2,8,Control
5,51487,Control,3,7,Control


In [11]:
# ============================================================
# CELL 8 — TOP-CONFIDENCE MEAN
# ============================================================

top_confidence_results = []

for sub_id, group in predictions_df.groupby("SUB_ID"):

    probabilities = group["SLICE_PROBABILITY"].values
    true_label = group["LABEL"].iloc[0]

    # Confidence = distance from the 0.5 decision boundary
    confidence = np.abs(probabilities - 0.5)

    # Select the 5 most confident slices
    top_indices = np.argsort(confidence)[-5:]

    top_probabilities = probabilities[top_indices]

    # Aggregate the top-confidence slices
    top_confidence_probability = np.mean(top_probabilities)

    # Subject-level prediction
    top_confidence_prediction = (
        "ASD"
        if top_confidence_probability >= 0.5
        else "Control"
    )

    top_confidence_results.append({
        "SUB_ID": sub_id,
        "TRUE_LABEL": true_label,
        "TOP_5_PROBABILITY": top_confidence_probability,
        "PREDICTION": top_confidence_prediction
    })

top_confidence_df = pd.DataFrame(top_confidence_results)

print("✅ Top-Confidence Mean results:")
display(top_confidence_df)

✅ Top-Confidence Mean results:


,SUB_ID,TRUE_LABEL,TOP_5_PROBABILITY,PREDICTION
0,51460,ASD,0.463312,Control
1,51463,ASD,0.495636,Control
2,51466,ASD,0.493026,Control
3,51475,Control,0.462581,Control
4,51476,Control,0.456518,Control
5,51487,Control,0.483493,Control


In [12]:
# ============================================================
# CELL 9 — TRIMMED MEAN
# ============================================================

trimmed_results = []

for sub_id, group in predictions_df.groupby("SUB_ID"):

    probabilities = np.sort(
        group["SLICE_PROBABILITY"].values
    )

    true_label = group["LABEL"].iloc[0]

    # Remove the lowest and highest prediction
    trimmed_probabilities = probabilities[1:-1]

    # Mean of the remaining 8 slices
    trimmed_probability = np.mean(trimmed_probabilities)

    trimmed_prediction = (
        "ASD"
        if trimmed_probability >= 0.5
        else "Control"
    )

    trimmed_results.append({
        "SUB_ID": sub_id,
        "TRUE_LABEL": true_label,
        "TRIMMED_PROBABILITY": trimmed_probability,
        "PREDICTION": trimmed_prediction
    })

trimmed_df = pd.DataFrame(trimmed_results)

print("✅ Trimmed Mean results:")
display(trimmed_df)

✅ Trimmed Mean results:


,SUB_ID,TRUE_LABEL,TRIMMED_PROBABILITY,PREDICTION
0,51460,ASD,0.485060,Control
1,51463,ASD,0.502403,ASD
2,51466,ASD,0.496456,Control
3,51475,Control,0.479240,Control
4,51476,Control,0.476952,Control
5,51487,Control,0.492189,Control


In [13]:
# ============================================================
# CELL 10 — FINAL EXPERIMENT 08 COMPARISON
# ============================================================

# Calculate accuracies
mean_accuracy = (
    mean_median_df["TRUE_LABEL"] == mean_median_df["MEAN_PREDICTION"]
).mean()

median_accuracy = (
    mean_median_df["TRUE_LABEL"] == mean_median_df["MEDIAN_PREDICTION"]
).mean()

majority_accuracy = (
    majority_df["TRUE_LABEL"] == majority_df["MAJORITY_PREDICTION"]
).mean()

top_confidence_accuracy = (
    top_confidence_df["TRUE_LABEL"] == top_confidence_df["PREDICTION"]
).mean()

trimmed_accuracy = (
    trimmed_df["TRUE_LABEL"] == trimmed_df["PREDICTION"]
).mean()


# Create final comparison table
aggregation_results = pd.DataFrame([
    {
        "AGGREGATION_METHOD": "Mean",
        "ACCURACY": mean_accuracy,
        "CORRECT_SUBJECTS": int(mean_accuracy * 6),
        "TOTAL_SUBJECTS": 6
    },
    {
        "AGGREGATION_METHOD": "Median",
        "ACCURACY": median_accuracy,
        "CORRECT_SUBJECTS": int(median_accuracy * 6),
        "TOTAL_SUBJECTS": 6
    },
    {
        "AGGREGATION_METHOD": "Majority Voting",
        "ACCURACY": majority_accuracy,
        "CORRECT_SUBJECTS": int(majority_accuracy * 6),
        "TOTAL_SUBJECTS": 6
    },
    {
        "AGGREGATION_METHOD": "Top-Confidence Mean",
        "ACCURACY": top_confidence_accuracy,
        "CORRECT_SUBJECTS": int(top_confidence_accuracy * 6),
        "TOTAL_SUBJECTS": 6
    },
    {
        "AGGREGATION_METHOD": "Trimmed Mean",
        "ACCURACY": trimmed_accuracy,
        "CORRECT_SUBJECTS": int(trimmed_accuracy * 6),
        "TOTAL_SUBJECTS": 6
    }
])

print("============================================================")
print("EXPERIMENT 08 — FINAL AGGREGATION COMPARISON")
print("============================================================")
print()

display(aggregation_results)

# Save results
EXP8_DIR = "/content/drive/MyDrive/ASD_XAI_Project/experiment_08_squeezenet_aggregation"

# Create Experiment 08 folder
import os
os.makedirs(EXP8_DIR, exist_ok=True)

# Save comparison
aggregation_results.to_csv(
    f"{EXP8_DIR}/aggregation_comparison_results.csv",
    index=False
)

# Save individual detailed results
mean_median_df.to_csv(
    f"{EXP8_DIR}/mean_median_results.csv",
    index=False
)

majority_df.to_csv(
    f"{EXP8_DIR}/majority_voting_results.csv",
    index=False
)

top_confidence_df.to_csv(
    f"{EXP8_DIR}/top_confidence_results.csv",
    index=False
)

trimmed_df.to_csv(
    f"{EXP8_DIR}/trimmed_mean_results.csv",
    index=False
)

print()
print("✅ Experiment 08 results saved successfully!")
print("Saved to:", EXP8_DIR)

EXPERIMENT 08 — FINAL AGGREGATION COMPARISON



,AGGREGATION_METHOD,ACCURACY,CORRECT_SUBJECTS,TOTAL_SUBJECTS
0,Mean,0.666667,4,6
1,Median,0.666667,4,6
2,Majority Voting,0.666667,4,6
3,Top-Confidence Mean,0.500000,3,6
4,Trimmed Mean,0.666667,4,6



✅ Experiment 08 results saved successfully!
Saved to: /content/drive/MyDrive/ASD_XAI_Project/experiment_08_squeezenet_aggregation
